# Modelos de predicción

En este capítulo predecimos el precio. Todo se evalúa por **zonas de la ciudad**: el modelo se
prueba en zonas que no vio al entrenar, porque así se usaría para valorar una vivienda nueva y
porque el capítulo 5 mostró que el precio de una vivienda se parece mucho al de sus vecinas.

Medimos cada modelo con cinco métricas: el R², el error absoluto medio (MAE), la raíz del error
cuadrático medio (RMSE), el error absoluto mediano (los tres en millones de pesos de 2025) y el
error porcentual medio (MAPE). **El MAPE es el criterio para elegir**, porque mide el error en
proporción al precio y trata igual a una vivienda barata que a una cara.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")   # evita que XGBoost y scikit-learn se bloqueen en Jupyter
from eda_utils import *
import time
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, ParameterSampler
from sklearn.neighbors import BallTree
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor

## Ventas registradas al avalúo

Al revisar dónde fallaba una primera versión del modelo encontramos que muchas ventas no
registran el precio de mercado sino el avalúo catastral: el precio, llevado a pesos de 2022 (el
año del avalúo), queda a menos de 10% del avalúo. Pasa sobre todo en la fuente `igac`.

In [2]:
d = viv[viv.lat.notna() & viv.lon.notna() & viv.area.notna()].copy().reset_index(drop=True)
a_2022 = ipc_anio.loc[2022] / ipc_anio
d["razon_avaluo"] = d.precio * d.anio.map(a_2022) / d.avaluo
d["al_avaluo"] = d.razon_avaluo.between(0.9, 1.1).fillna(False).astype(bool)   # sin avalúo: no se puede saber
d["tipo_txt"] = np.where((d.condicion_predio == "NPH").fillna(False), "casa", "apartamento u otro")

(d.groupby(["tipo_txt", "fuente"])
  .agg(ventas=("precio", "size"), precio_sobre_avaluo=("razon_avaluo", "median"),
       pct_al_avaluo=("al_avaluo", "mean"))
  .assign(pct_al_avaluo=lambda t: (t.pct_al_avaluo * 100).round(1)).round(2))

ventas  precio_sobre_avaluo  pct_al_avaluo
tipo_txt           fuente                                                       
apartamento u otro avisos_registro        83                 1.63          12.00
                   georreferenciadas    2113                 1.75           1.50
                   igac                18780                 1.32          15.30
                   observatorio         7975                 1.43           1.10
casa               avisos_registro        33                 1.89           3.00
                   georreferenciadas      46                 0.97          13.00
                   igac                 2685                 1.00          55.80
                   observatorio         2866                 1.82           5.20

En las casas del IGAC el precio mediano es igual al avalúo (1,00), y el 57 % de las que tienen avalúo queda a menos de 10 % de él. En el observatorio esto solo pasa con el 5 % de las casas y el 1 % de los apartamentos. Esas ventas no reflejan el mercado, porque lo más probable es que en la escritura se haya puesto el avalúo y no lo que realmente se pagó, así que las quitamos del modelo. Las ventas sin avalúo se dejan, porque en ellas no hay forma de saberlo.

Los apartamentos del IGAC también tienen este problema. De los 9.692 que tienen avalúo, el 30 % queda a menos de 10 % de él (la tabla muestra 15 % porque lo calcula sobre todos, incluso los que no tienen avalúo). Esto no se debe a que sean obras nuevas, ya que pasa igual o más en apartamentos de más de 10 años. Hay otros 9.087 apartamentos del IGAC sin avalúo que no se pueden revisar, y es probable que una parte de ellos también tenga este problema. 


In [3]:
print(f"Ventas registradas al avalúo: {d.al_avaluo.sum():,} de {len(d):,} ({d.al_avaluo.mean():.1%})")
d = d[~d.al_avaluo].reset_index(drop=True)
print(f"Base para modelar: {len(d):,} ventas")

Ventas registradas al avalúo: 4,650 de 34,581 (13.4%)
Base para modelar: 29,931 ventas


## **Variables y evaluación**

Las variables son el área (en logaritmo), habitaciones, baños, piso, plantas, edad, estrato,
latitud, longitud, año de venta y tipo de predio. La fuente del dato solo se usa al entrenar, para corregir que cada fuente registra el precio a un nivel distinto, y al valorar una vivienda nueva se fija siempre en el observatorio, así que nadie tiene que darla. La variable objetivo es el logaritmo del precio en pesos de 2025.

Dividimos la ciudad en bloques de 0,01° (más o menos 1,1 km de lado). Algunos bloques escogidos al azar, que suman cerca del 20 % de las ventas, se dejan aparte como prueba final y solo se usan una vez, al terminar. Con el resto de los bloques comparamos los modelos usando validación cruzada de cinco particiones, también por bloques. Los bloques de Sector Alameda del Río (unas 4.430 ventas en cuatro bloques) siempre se quedan en el entrenamiento, porque si cayeran en la prueba, gran parte de ella sería un solo proyecto y sería mucho más fácil de predecir que el resto de la ciudad.

In [4]:
d["log_area"] = np.log10(d.area)
d["log_pm2"] = np.log10(d.precio_m2_real)
d["estrato_n"] = d.estrato.astype(str).map({e: i + 1 for i, e in enumerate(ESTRATOS)})
for t in ["PH_Unidad_Predial", "NPH", "PH_Matriz"]:
    d[f"tipo_{t}"] = (d.condicion_predio == t).astype(float)
d["bloque"] = (np.floor(d.lat / 0.01).astype(int).astype(str) + "_" +
               np.floor(d.lon / 0.01).astype(int).astype(str))

for f in ["igac", "observatorio", "georreferenciadas"]:
    d[f"fuente_{f}"] = (d.fuente == f).astype(float)

# La fuente del dato se usa solo para entrenar: cada fuente registra el precio a un nivel distinto
# (el IGAC, más bajo) y así el modelo lo corrige. Al valorar una vivienda nueva la fuente se fija
# en la de referencia, el observatorio, así que nadie tiene que darla.
FUENTES = ["fuente_igac", "fuente_observatorio", "fuente_georreferenciadas"]
REFERENCIA = "observatorio"
BASE = ["log_area", "habitaciones", "banios", "piso", "plantas", "edad", "estrato_n", "lat", "lon",
        "anio", "tipo_PH_Unidad_Predial", "tipo_NPH", "tipo_PH_Matriz"] + FUENTES

bloques = d.groupby("bloque").agg(ventas=("bloque", "size"),
                                  alameda=("barrio", lambda s: (s == "SECTOR ALAMEDA DEL RIO").mean()))
candidatos = bloques[bloques.alameda < 0.5].sample(frac=1, random_state=42)
bloques_prueba = candidatos.index[candidatos.ventas.cumsum() <= 0.2 * len(d)]
es_prueba = d.bloque.isin(bloques_prueba)
ent, prueba = d[~es_prueba].reset_index(drop=True), d[es_prueba].reset_index(drop=True)
y_ent, y_prueba = ent.log_precio.to_numpy(), prueba.log_precio.to_numpy()
print(f"Entrenamiento: {len(ent):,} ventas en {ent.bloque.nunique()} bloques")
print(f"Prueba final : {len(prueba):,} ventas en {prueba.bloque.nunique()} bloques")

Entrenamiento: 24,286 ventas en 71 bloques
Prueba final : 5,645 ventas en 24 bloques


## Comparación de modelos

In [5]:
MODELOS = {
    "Regresión lineal": lambda: make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                              LinearRegression()),
    # sin imputar: desde scikit-learn 1.4 Random Forest maneja los vacíos (p. ej. el piso
    # de las ventas sin unidad propia) igual que XGBoost, sin inventar un valor
    "Random Forest": lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                                   max_features=0.5, n_jobs=4, random_state=42),
    "Gradient Boosting": lambda: HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05,
                                                               min_samples_leaf=20, random_state=42),
    "XGBoost": lambda: XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                                    colsample_bytree=0.8, min_child_weight=3, n_jobs=4, random_state=42),
}
CV = GroupKFold(5)

def metricas(y, p):
    """Cinco métricas. y y p son el log10 del precio en pesos de 2025."""
    real, pred = 10.0 ** y / 1e6, 10.0 ** p / 1e6
    return {"R²": 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "MAE (M)": np.mean(np.abs(real - pred)),
            "RMSE (M)": np.sqrt(np.mean((real - pred) ** 2)),
            "error abs. mediano (M)": np.median(np.abs(real - pred)),
            "MAPE (%)": np.mean(np.abs(pred / real - 1)) * 100}

def validar(fabrica, columnas, datos):
    """Predicciones fuera de muestra con validación cruzada por bloques."""
    pred = np.empty(len(datos))
    for tr, va in CV.split(datos, groups=datos.bloque):
        a, b = datos.iloc[tr], datos.iloc[va]
        pred[va] = fabrica().fit(a[columnas], a.log_precio).predict(b[columnas])
    return pred

filas = []
for nombre, fabrica in MODELOS.items():
    filas.append({"modelo": nombre, **metricas(y_ent, validar(fabrica, BASE, ent))})
pd.DataFrame(filas).set_index("modelo").round(3)

,R²,MAE (M),RMSE (M),error abs. mediano (M),MAPE (%)
modelo,,,,,
Regresión lineal,0.62,98.94,725.52,55.48,37.30
Random Forest,0.65,93.51,307.03,48.20,32.30
Gradient Boosting,0.67,89.68,290.33,48.93,32.04
XGBoost,0.64,92.86,281.30,51.26,32.42


Los tres modelos de árboles quedan cerca entre sí. Gradient Boosting tiene el menor MAPE (32,0%), seguido de Random Forest (32,3%) y XGBoost (32,4%), y también el mayor R² (0,67) y el menor MAE (90 M). La regresión lineal queda bastante atrás, con un MAPE de 37,3% y un RMSE de 726 M, porque se equivoca mucho en algunas viviendas caras. Por eso elegimos **Gradient Boosting** para el resto del capítulo. Los tres modelos de árboles están con su configuración inicial; en la sección de ajuste se afina Gradient Boosting.

## Variables del entorno

Probamos también variables que describen qué hay alrededor de cada vivienda. Salen de
OpenStreetMap.

- la distancia (km) al río Magdalena, a la vía principal más cercana, a la parada de Transmetro,
  al centro comercial, a la universidad y a la clínica u hospital más cercanos;
- cuántos colegios, parques, supermercados y centros de salud hay a menos de 1 km;
- cuántas ventas hay en la misma manzana, que indica si es un edificio o conjunto grande.

In [6]:
R_TIERRA_KM = 6371
osm = pd.read_csv("datos/entorno_osm.csv")
osm = osm[(osm.categoria != "rio") | (osm.nombre == "Río Magdalena")]   # sin los caños
DISTANCIAS = {"rio": "dist_rio", "via_principal": "dist_via", "transmetro": "dist_transmetro",
              "centro_comercial": "dist_centro_comercial", "universidad": "dist_universidad",
              "salud": "dist_salud"}
CONTEOS = {"colegio": "colegios_1km", "parque": "parques_1km", "supermercado": "supermercados_1km",
           "salud": "salud_1km"}
ventas_manzana = d.manzana.value_counts()

def entorno(datos):
    xy = np.radians(datos[["lat", "lon"]].to_numpy())
    salida = pd.DataFrame(index=datos.index)
    for cat, col in DISTANCIAS.items():
        arbol = BallTree(np.radians(osm.loc[osm.categoria == cat, ["lat", "lon"]].to_numpy()), metric="haversine")
        salida[col] = arbol.query(xy, k=1)[0][:, 0] * R_TIERRA_KM
    for cat, col in CONTEOS.items():
        arbol = BallTree(np.radians(osm.loc[osm.categoria == cat, ["lat", "lon"]].to_numpy()), metric="haversine")
        salida[col] = arbol.query_radius(xy, r=1 / R_TIERRA_KM, count_only=True)
    salida["ventas_manzana"] = np.log10(datos.manzana.map(ventas_manzana))
    return salida

OSM = list(DISTANCIAS.values()) + list(CONTEOS.values()) + ["ventas_manzana"]
ent, prueba = ent.join(entorno(ent)), prueba.join(entorno(prueba))
filas = []
for nombre in ["Random Forest", "Gradient Boosting", "XGBoost"]:
    for etiqueta, cols in [("vivienda + ubicación", BASE), ("+ entorno", BASE + OSM)]:
        filas.append({"modelo": nombre, "variables": etiqueta, **metricas(y_ent, validar(MODELOS[nombre], cols, ent))})
tabla_entorno = pd.DataFrame(filas)
x = tabla_entorno[tabla_entorno.modelo == "Gradient Boosting"].set_index("variables")["MAPE (%)"]
USAR_ENTORNO = bool(x["+ entorno"] < x["vivienda + ubicación"])
print("¿Usar el entorno en el modelo final?", "sí" if USAR_ENTORNO else "no")
tabla_entorno.round(3)

¿Usar el entorno en el modelo final? sí


,modelo,variables,R²,MAE (M),RMSE (M),error abs. mediano (M),MAPE (%)
0,Random Forest,vivienda + ubicación,0.65,93.51,307.03,48.20,32.30
1,Random Forest,+ entorno,0.63,97.21,311.67,51.21,32.91
2,Gradient Boosting,vivienda + ubicación,0.67,89.68,290.33,48.93,32.04
3,Gradient Boosting,+ entorno,0.66,92.19,297.63,50.05,31.80
4,XGBoost,vivienda + ubicación,0.64,92.86,281.30,51.26,32.42
5,XGBoost,+ entorno,0.60,97.41,301.87,53.05,34.01


Con el entorno, el MAPE de Gradient Boosting baja de 32,0% a 31,8%, así que el modelo final sí usa estas variables. La mejora es pequeña y solo en el MAPE: el MAE, el RMSE y el error absoluto mediano empeoran un poco. A Random Forest y XGBoost el entorno los empeora (32,3% a 32,9% y 32,4% a 34,0%).

## Ajuste de hiperparámetros

Probamos 25 combinaciones de Gradient Boosting con la misma validación por bloques y nos quedamos
con la de menor MAPE, solo si supera a la configuración inicial.

In [7]:
COLUMNAS = BASE + OSM if USAR_ENTORNO else BASE
espacio = {"learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1], "max_iter": [300, 600, 1000],
           "max_leaf_nodes": [15, 31, 63], "max_depth": [None, 4, 6, 8],
           "min_samples_leaf": [10, 20, 50, 100], "l2_regularization": [0.0, 0.1, 1.0, 10.0],
           "max_features": [0.5, 0.8, 1.0]}
inicial = dict(max_iter=600, learning_rate=0.05, max_leaf_nodes=31, max_depth=None,
               min_samples_leaf=20, l2_regularization=0.0, max_features=1.0)
pruebas = []
for params in [inicial] + list(ParameterSampler(espacio, n_iter=25, random_state=42)):
    p = validar(lambda: HistGradientBoostingRegressor(random_state=42, **params), COLUMNAS, ent)
    pruebas.append({"params": params, **params, **metricas(y_ent, p), "inicial": params is inicial})
busqueda = pd.DataFrame(pruebas).sort_values("MAPE (%)")
mejores = {k: (v.item() if hasattr(v, "item") else v) for k, v in busqueda.iloc[0]["params"].items()}
print("Configuración elegida:", mejores)
busqueda.drop(columns="params").head(5).round(3)

Configuración elegida: {'min_samples_leaf': 100, 'max_leaf_nodes': 31, 'max_iter': 300, 'max_features': 0.5, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 1.0}


,max_iter,learning_rate,max_leaf_nodes,max_depth,min_samples_leaf,l2_regularization,max_features,R²,MAE (M),RMSE (M),error abs. mediano (M),MAPE (%),inicial
4,300,0.05,31,NaN,100,1.00,0.50,0.68,89.31,305.14,47.02,30.93,False
19,600,0.02,31,6.00,100,1.00,0.50,0.68,89.91,304.93,47.38,31.01,False
12,600,0.03,15,8.00,20,0.00,0.50,0.67,89.36,298.68,47.96,31.14,False
13,300,0.03,15,NaN,20,10.00,0.80,0.69,89.28,299.55,47.01,31.15,False
21,1000,0.10,15,8.00,10,1.00,0.50,0.67,90.67,298.93,45.78,31.22,False


La mejor combinación usa 300 árboles con una tasa de aprendizaje de 0,05, hasta 31 hojas por árbol, al menos 100 ventas por hoja, una regularización de 1 y la mitad de las variables en cada división. Baja el MAPE en validación de 31,8% a 30,9%, con un R² de 0,68. Es una configuración más prudente que la inicial: hojas con más ventas y menos variables por división, lo que evita que el modelo se aprenda casos particulares.

## Prueba final

Entrenamos el modelo elegido con todo el conjunto de entrenamiento y lo evaluamos una sola vez
en las zonas que dejamos aparte al principio. La regresión lineal queda como referencia.

In [8]:
final = HistGradientBoostingRegressor(random_state=42, **mejores).fit(ent[COLUMNAS], ent.log_precio)
p_final = final.predict(prueba[COLUMNAS])
p_lineal = MODELOS["Regresión lineal"]().fit(ent[BASE], ent.log_precio).predict(prueba[BASE])
pd.DataFrame({"Gradient Boosting (modelo elegido)": metricas(y_prueba, p_final),
              "Regresión lineal": metricas(y_prueba, p_lineal)}).T.round(3)

,R²,MAE (M),RMSE (M),error abs. mediano (M),MAPE (%)
Gradient Boosting (modelo elegido),0.61,57.16,104.83,34.86,29.85
Regresión lineal,0.56,64.63,240.62,31.15,35.56


In [9]:
prueba["error_pct"] = (10 ** (p_final - y_prueba) - 1) * 100
prueba["error_abs_M"] = np.abs(10 ** p_final - 10 ** y_prueba) / 1e6
resumen = lambda g: g.agg(ventas=("error_pct", "size"),
                          **{"MAPE (%)": ("error_pct", lambda e: np.mean(np.abs(e))),
                             "error abs. mediano (M)": ("error_abs_M", "median")}).round(1)
pd.concat([resumen(prueba.groupby("tipo_txt")),
           resumen(prueba.assign(loc=prueba.localidad.fillna("Sin localidad").str.title()).groupby("loc"))],
          keys=["Tipo", "Localidad"])

ventas  MAPE (%)  error abs. mediano (M)
Tipo      apartamento u otro          4691     26.50                   31.60
          casa                         954     46.20                   52.80
Localidad Metropolitana                308     46.20                   35.00
          Norte - Centro Historico    2142     34.00                   59.40
          Riomar                        29     52.50                   93.90
          Rural                       1302     16.30                    6.50
          Sin Localidad                 39     40.50                   38.00
          Suroccidente                1308     28.20                   26.00
          Suroriente                   517     39.10                   40.70

En las zonas de prueba Gradient Boosting tiene un MAPE de 29,9%, frente a 35,6% de la regresión lineal, y un R² de 0,61 frente a 0,56. También tiene un MAE y un RMSE mucho menores (57 M y 105 M, frente a 65 M y 241 M). Solo el error absoluto mediano es algo menor en la regresión lineal (31 M frente a 35 M). Son solo 24 zonas, así que el capítulo 8 repite la comparación con 20 divisiones.

El error se concentra en las casas (MAPE de 46%, frente a 27% en los apartamentos). Por zonas, el modelo acierta mejor en la zona rural (16%), donde hay grandes proyectos de apartamentos parecidos, y peor en Metropolitana y Suroriente (46% y 39%).

### Cuánto influye cada variable en el modelo

Para medir cuánto depende el modelo de cada variable, desordenamos sus valores en las ventas de
prueba (como si fueran de otra vivienda) y vemos cuántos puntos sube el MAPE. Si el error sube
mucho, el modelo depende mucho de esa variable; si casi no cambia, la variable influye poco. Las
tres columnas del tipo de predio se desordenan juntas, igual que las de la fuente.

In [10]:
GRUPOS = {"área": ["log_area"], "habitaciones": ["habitaciones"], "baños": ["banios"],
          "piso": ["piso"], "plantas": ["plantas"], "edad": ["edad"], "estrato": ["estrato_n"],
          "latitud": ["lat"], "longitud": ["lon"], "año de venta": ["anio"],
          "tipo de predio": ["tipo_PH_Unidad_Predial", "tipo_NPH", "tipo_PH_Matriz"],
          "fuente": FUENTES}
if USAR_ENTORNO:
    GRUPOS["entorno"] = OSM
mape = lambda p: np.mean(np.abs(10.0 ** (p - y_prueba) - 1)) * 100
mape_base = mape(final.predict(prueba[COLUMNAS]))
rng = np.random.default_rng(42)
influencia = {}
for nombre, cols in GRUPOS.items():
    aumentos = []
    for _ in range(5):                       # cinco desórdenes distintos, para promediar el azar
        x = prueba[COLUMNAS].copy()
        x[cols] = x[cols].astype(float).to_numpy()[rng.permutation(len(x))]   # vacíos como NaN
        aumentos.append(mape(final.predict(x)) - mape_base)
    influencia[nombre] = np.mean(aumentos)
(pd.Series(influencia).sort_values(ascending=False).round(2)
   .to_frame("aumento del MAPE al desordenarla (puntos)"))

,aumento del MAPE al desordenarla (puntos)
área,9.33
entorno,5.55
estrato,4.33
latitud,3.16
fuente,1.75
piso,1.09
longitud,1.09
tipo de predio,0.29
edad,0.27
año de venta,0.26


El área es por lejos la variable que más influye: si se desordena, el MAPE sube 9,3 puntos. Le siguen las variables del entorno (5,6), el estrato (4,3) y la latitud (3,2). La fuente también pesa (1,8), lo que confirma que el modelo la usa para corregir el nivel de precio de cada fuente. El piso y la longitud influyen poco (1,1), y las demás casi nada. Que las habitaciones y los baños influyan tan poco no quiere decir que no importen, sino que el área ya lleva casi toda esa información.

### Cómo se valora una vivienda nueva

El modelo se entrena con la fuente de cada venta, pero quien valora su vivienda no tiene una fuente
que dar. Por eso, al predecir, la fuente se fija siempre en la de referencia, el observatorio. La
tabla compara, para cinco ventas de prueba del IGAC, el precio registrado, la predicción con su
propia fuente y el valor que recibiría una vivienda nueva con las mismas características.

In [11]:
def a_referencia(datos):
    """Fija la fuente en la de referencia, como se haría con una vivienda que no viene de ninguna fuente."""
    datos = datos.copy()
    for f in FUENTES:
        datos[f] = float(f == f"fuente_{REFERENCIA}")
    return datos

ejemplo = prueba[prueba.fuente == "igac"].sample(5, random_state=1)
display(pd.DataFrame({
    "barrio": ejemplo.barrio.str.title(), "área (m²)": ejemplo.area.round(0),
    "estrato": ejemplo.estrato_n,
    "precio registrado (M)": (10 ** ejemplo.log_precio / 1e6).round(0),
    "predicción con su fuente (M)": (10 ** final.predict(ejemplo[COLUMNAS]) / 1e6).round(0),
    "valor de una vivienda nueva (M)": (10 ** final.predict(a_referencia(ejemplo)[COLUMNAS]) / 1e6).round(0),
}).set_index("barrio"))

# cuánto cambia, en promedio, la predicción al fijar la fuente en la de referencia
cambio = 10 ** (final.predict(a_referencia(prueba)[COLUMNAS]) - final.predict(prueba[COLUMNAS])) - 1
(pd.Series(cambio * 100, index=prueba.index).groupby(prueba.fuente).median().round(1)
   .rename("cambio mediano de la predicción (%)").to_frame())

,área (m²),estrato,precio registrado (M),predicción con su fuente (M),valor de una vivienda nueva (M)
barrio,,,,,
Las Delicias,67.00,5.00,281.00,233.00,253.00
Rural,51.00,1.00,147.00,116.00,129.00
Rural,42.00,1.00,86.00,90.00,94.00
Carrizal,158.00,2.00,118.00,107.00,163.00
Villas De San Pablo,58.00,1.00,129.00,102.00,109.00


,cambio mediano de la predicción (%)
fuente,
avisos_registro,10.20
georreferenciadas,0.70
igac,10.50
observatorio,0.00


Al fijar la fuente en el observatorio, las predicciones de las ventas del IGAC suben un 10,5% en la mediana, las de los avisos de registro un 10,2% y las georreferenciadas un 0,7%. Ese es el ajuste que el modelo aprendió por el nivel de precio de cada fuente. Así, una vivienda nueva se valora siempre al nivel del observatorio. Por ejemplo, el apartamento de 158 m² de Carrizal se registró en el IGAC por 118 M, el modelo lo predice en 107 M con su fuente y lo valora en 163 M como vivienda nueva.

## Resumen

- El 13% de las ventas registra el avalúo en lugar del precio de mercado: el 57% de las casas del IGAC con avalúo y el 30% de sus apartamentos. Las quitamos del modelo. Otros 9.087 apartamentos del IGAC no tienen avalúo y no se pueden revisar.
- Con el MAPE como criterio, el mejor modelo es Gradient Boosting con las variables del entorno y los hiperparámetros ajustados: MAPE de 30,9% en validación por zonas (R² de 0,68) y de 29,9% en la prueba final, frente a 35,6% de la regresión lineal.
- La fuente del dato se usa solo para entrenar. Al valorar una vivienda nueva se fija en el observatorio, y en las ventas del IGAC eso sube la predicción un 10,5%.
- El área es la variable que más influye, seguida del entorno, el estrato y la latitud.
- El error se concentra en las casas (46%, frente a 27% en los apartamentos). Al precio de una casa le pesan mucho el área del lote, el estado y las remodelaciones, y nada de eso está en la base.

